# RAG — Phase 1: Ingest → Chunk → Store → Retrieve

This notebook builds the Phase-1 pipeline from `RAG.md`:

1. Load documents (PDFs, Markdown, webpages)
2. Chunk them (500–800 tokens, 100-token overlap)
3. Embed and store chunks in a vector store (Chroma)
4. Retrieve the top-K most relevant chunks for a query and generate a cited answer

LLM generation uses **Groq**. Groq doesn't serve an embeddings endpoint, so embeddings run locally via `sentence-transformers` — free, no extra API key, and good enough for Phase-1. You can swap this for a hosted embedding model later without touching the rest of the pipeline.

## 1. Setup & environment

We need a Groq API key (get one at console.groq.com) stored in a `.env` file as `GROQ_API_KEY=...` — never hardcode it in the notebook. `python-dotenv` loads it into the environment at runtime.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
assert GROQ_API_KEY, "Set GROQ_API_KEY in a .env file in this project's root."
print("Groq key loaded:", GROQ_API_KEY[:6] + "..." if GROQ_API_KEY else "MISSING")

# WebBaseLoader (used in §2) sends an HTTP User-Agent header; some sites
# deprioritize or block requests with none set. Set it once, here.
os.environ.setdefault("USER_AGENT", "rag-phase1-notebook/1.0")

Groq key loaded: gsk_0d...


'rag-phase1-notebook/1.0'

## 2. Load documents

Three source types, three loaders:
- **PDFs** → `PyPDFLoader` (one `Document` per page, keeps page number in metadata — useful for citations later)
- **Markdown** → `UnstructuredMarkdownLoader`
- **Webpages** → `WebBaseLoader`

Per the Explorer screenshot, `data/` sits next to `notebook_demo/`, not inside it — so if this notebook runs from `notebook_demo/`, the path is `../data`, not `data`. We walk that folder, dispatch by extension, and tag every `Document` with its source path so citations can point back to the right file.

If you move the notebook or run it from a different working directory, adjust `DATA_DIR` accordingly — or make it robust to cwd with `Path(__file__).parent / ".." / "data"` equivalents (not available in notebooks) or by anchoring on `Path.cwd()`.

In [2]:
from pathlib import Path
from langchain_community.document_loaders import (
    PyPDFLoader,
    UnstructuredMarkdownLoader,
    WebBaseLoader,
)

DATA_DIR = Path("../data")       # data/ is a sibling of notebook_demo/, not inside it
WEB_URLS = [
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"
]

def load_local_documents(data_dir: Path):
    docs = []
    for path in data_dir.rglob("*"):
        if path.suffix.lower() == ".pdf":
            loader = PyPDFLoader(str(path))
        elif path.suffix.lower() in (".md", ".markdown"):
            loader = UnstructuredMarkdownLoader(str(path))
        else:
            continue
        loaded = loader.load()
        for d in loaded:
            d.metadata["source"] = str(path)
        docs.extend(loaded)
    return docs

def load_web_documents(urls):
    if not urls:
        return []
    loader = WebBaseLoader(urls)
    docs = loader.load()
    for d in docs:
        d.metadata["source"] = d.metadata.get("source", "web")
    return docs

raw_docs = load_local_documents(DATA_DIR) + load_web_documents(WEB_URLS)
print(f"Loaded {len(raw_docs)} raw documents")
if raw_docs:
    print("Example metadata:", raw_docs[0].metadata)

C:\Users\smrut\AppData\Local\Temp\ipykernel_22472\2443879791.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


Loaded 3 raw documents
Example metadata: {'source': '..\\data\\RAG.md'}


## 3. Chunk documents (500–800 tokens, 100-token overlap)

Two design choices matter here:

- **Measuring in tokens, not characters.** `RecursiveCharacterTextSplitter` counts length however you tell it to. We pass a `tiktoken`-based length function so "500–800" means actual model tokens, not a rough character proxy.
- **Overlap.** 100 tokens of overlap means a sentence split across a chunk boundary still appears whole in at least one chunk, so retrieval doesn't lose context that straddles a cut point.

We split on paragraph → sentence → word boundaries in that priority order, which keeps chunks semantically coherent instead of cutting mid-sentence whenever possible.

In [3]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

encoding = tiktoken.get_encoding("cl100k_base")

def token_len(text: str) -> int:
    return len(encoding.encode(text))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,          # midpoint of the 500-800 target range
    chunk_overlap=100,
    length_function=token_len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(raw_docs)
print(f"Created {len(chunks)} chunks from {len(raw_docs)} documents")
if chunks:
    print("Example chunk token count:", token_len(chunks[0].page_content))
    print(chunks[0].page_content[:300])

Created 15 chunks from 3 documents
Example chunk token count: 267
RAG

Phase-1 ingest documents;pdfs,mds,webpages->chunk them to pieces;500-800 tokens;100 tokens overlap between chunks->store chunks in a vector store->build a retrieval pipeline that pulls the top K most relevant chunks for a given query and generates the answer that cites where the information cam


## 4. Embedding model

Groq is fast/cheap for *generation* but has no embeddings API, so chunks are embedded locally with `sentence-transformers/all-MiniLM-L6-v2` — a small, well-tested model that runs on CPU with no external calls. This keeps ingestion free and reproducible; swap it later for a hosted model if retrieval quality needs it.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# sanity check
test_vec = embedding_model.embed_query("sanity check")
print("Embedding dimension:", len(test_vec))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimension: 384


## 5. Store chunks in a vector store (ChromaDB)

Chroma persists to disk under `PERSIST_DIR`, so you only pay the embedding cost once — re-running this cell after the first successful run will just reopen the existing store unless you clear the directory.

In [5]:
from langchain_chroma import Chroma

PERSIST_DIR = "chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=PERSIST_DIR,
    collection_name="rag_phase1",
)

print(f"Vector store built with {vectorstore._collection.count()} chunks at ./{PERSIST_DIR}")

Vector store built with 15 chunks at ./chroma_db


## 6. Retrieval pipeline — top-K chunks for a query

`as_retriever` wraps the vector store's similarity search behind a standard LangChain interface. `k` is how many chunks come back per query — start at 5, tune later once you have eval data (Phase-3).

In [6]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# quick manual check before wiring up generation
test_query = "What is this project about?"   # replace with a real question about your data
retrieved = retriever.invoke(test_query)
for i, doc in enumerate(retrieved, 1):
    print(f"--- chunk {i} (source: {doc.metadata.get('source')}) ---")
    print(doc.page_content[:200], "\n")

--- chunk 1 (source: ..\data\resume.pdf) ---
CTTC Bhubaneswar
–Built and evaluated machine learning, deep learning, and NLP models in Python for predictive analytics and
classification tasks.
–Developed a computer vision–based face recognition s 

--- chunk 2 (source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation) ---
John von Neumann
Christopher D. Manning
Claude Shannon
Shun'ichi Amari
Kunihiko Fukushima
Takeo Kanade
Marvin Minsky
John McCarthy
Nathaniel Rochester
Allen Newell
Cliff Shaw
Herbert A. Simon
Oliver S 

--- chunk 3 (source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation) ---
Type of information retrieval using LLMs
Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data so 

--- chunk 4 (source: https://en.wikipedia.org/wiki/Retrieval-augmented_generation) ---
Category











Retrieved from "https://en.wikipedia.org/w/index.php?title=Retrie

## 7. Generate a cited answer with Groq

We use `openai/gpt-oss-120b` — Groq moved general API access to its Llama chat models (`llama-3.3-70b-versatile`, `llama-3.1-8b-instant`) behind an Enterprise plan, so a standard key now gets a 404 on those. GPT-OSS 120B is the current flagship on standard/free-tier keys, with a smaller/faster `openai/gpt-oss-20b` also available. If this 404s again later, check `https://console.groq.com/docs/models` — Groq's lineup shifts.

The prompt does two jobs at once: answer *only* from the retrieved chunks, and cite which source each claim came from. Each chunk is numbered and labeled with its source path so the model can point back to `[1]`, `[2]`, etc. Phase-2 will add hard citation *enforcement* (refuse to answer if chunks don't support it) — here we just ask for it in the prompt.

In [7]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    groq_api_key=GROQ_API_KEY,
)

PROMPT_TEMPLATE = """You are a helpful assistant answering questions using only the context below.
Cite the source number (e.g. [1]) after every claim you make. If the context does not contain
enough information to answer, say so explicitly instead of guessing.

Context:
{context}

Question: {question}

Answer (with citations):"""

def format_context(docs):
    lines = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "unknown")
        lines.append(f"[{i}] (source: {src})\n{d.page_content}")
    return "\n\n".join(lines)

def answer_query(question: str, k: int = 5):
    docs = vectorstore.similarity_search(question, k=k)
    context = format_context(docs)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    response = llm.invoke(prompt)
    return response.content, docs

## 8. Try it end-to-end

Replace the question below with something your ingested documents can actually answer.

In [8]:
question = "What is this project about?"
answer, sources = answer_query(question)

print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for i, d in enumerate(sources, 1):
    print(f"[{i}] {d.metadata.get('source')}")

ANSWER:
 The project involves building and evaluating machine‑learning, deep‑learning, and natural‑language‑processing models in Python for predictive‑analytics and classification tasks, and creating a computer‑vision‑based face‑recognition system that uses OpenCV together with deep‑learning techniques【1】.

SOURCES USED:
[1] ..\data\resume.pdf
[2] https://en.wikipedia.org/wiki/Retrieval-augmented_generation
[3] https://en.wikipedia.org/wiki/Retrieval-augmented_generation
[4] https://en.wikipedia.org/wiki/Retrieval-augmented_generation
[5] https://en.wikipedia.org/wiki/Retrieval-augmented_generation


## 9. Test scenarios

Phase-1 doesn't have automated evaluation yet — that's Phase-3 (golden Q&A set + `ragas` faithfulness scoring, wired into CI). Until then, these are quick manual sanity checks to catch obvious breakage: bad chunking, retrieval pulling the wrong document, or the model answering from outside the context. Run these after any change to loaders, chunking, or the vector store.

### 9.1 Chunk size sanity check

Confirms the splitter actually respects the 500–800 token target (with the last chunk of each document allowed to run shorter) and that overlap is non-zero. If most chunks are far outside this range, the `separators` list or `chunk_size` needs tuning for your document type.

In [9]:
import statistics

token_counts = [token_len(c.page_content) for c in chunks]

print(f"Chunks: {len(token_counts)}")
print(f"Min / median / max tokens: {min(token_counts)} / {statistics.median(token_counts)} / {max(token_counts)}")

# how many fall outside the 500-800 target (excluding short trailing chunks, which are expected)
outside_target = [t for t in token_counts if t > 800]
print(f"Chunks over 800 tokens (should be ~0): {len(outside_target)}")

# spot-check overlap: the end of chunk i should share text with the start of chunk i+1
# (only meaningful for chunks from the same source document)
if len(chunks) > 1 and chunks[0].metadata.get("source") == chunks[1].metadata.get("source"):
    tail = chunks[0].page_content[-100:]
    head = chunks[1].page_content[:200]
    shares_text = any(tail[-n:] in head for n in (20, 30, 40) if len(tail) >= n)
    print(f"Adjacent chunks share overlapping text: {shares_text}")

Chunks: 15
Min / median / max tokens: 178 / 525 / 694
Chunks over 800 tokens (should be ~0): 0


### 9.2 Retrieval sanity check — does the right document come back?

Ask a question that should only be answerable from one specific source file, and confirm that file actually shows up in the top-K results. If it doesn't, the problem is upstream of generation — bad chunking, wrong embedding model, or `k` set too low — not the LLM.

In [10]:
# Edit this to match a fact that only exists in one of your documents
test_query = "What does Phase-2 add to the RAG pipeline?"   # should hit RAG.md specifically
expected_source_substring = "RAG.md"

results = vectorstore.similarity_search(test_query, k=5)
sources_hit = [r.metadata.get("source", "") for r in results]

matched = any(expected_source_substring in s for s in sources_hit)
print(f"Query: {test_query!r}")
print(f"Sources retrieved: {sources_hit}")
print(f"Expected source present in top-5: {matched}")

Query: 'What does Phase-2 add to the RAG pipeline?'
Sources retrieved: ['https://en.wikipedia.org/wiki/Retrieval-augmented_generation', '..\\data\\RAG.md', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', '..\\data\\resume.pdf']
Expected source present in top-5: True


### 9.3 Out-of-scope query — does it admit what it doesn't know?

Ask something the ingested documents definitely don't cover. Phase-1's prompt *asks* the model to say so rather than guess, but nothing enforces it yet (that's Phase-2's citation enforcement). This test just tells you how often the current prompt already behaves honestly, as a baseline to compare against once enforcement is added.

In [11]:
off_topic_question = "What is the capital of Mongolia?"  # unrelated to any ingested document

answer, sources = answer_query(off_topic_question)
print("ANSWER:\n", answer)
print("\nRetrieved sources (for context, even though irrelevant):")
for d in sources:
    print(" -", d.metadata.get("source"))

# Rough automated check: did the model hedge instead of confidently answering?
hedge_phrases = ["don't have", "does not contain", "no information", "cannot answer", "not mentioned", "not provided"]
hedged = any(p in answer.lower() for p in hedge_phrases)
print(f"\nModel appears to have hedged appropriately: {hedged}")

ANSWER:
 I’m sorry, but the provided context does not contain any information about the capital of Mongolia, so I cannot answer the question from the given sources.

Retrieved sources (for context, even though irrelevant):
 - ..\data\resume.pdf
 - https://en.wikipedia.org/wiki/Retrieval-augmented_generation
 - ..\data\resume.pdf
 - https://en.wikipedia.org/wiki/Retrieval-augmented_generation
 - https://en.wikipedia.org/wiki/Retrieval-augmented_generation

Model appears to have hedged appropriately: True


### 9.4 Batch run over multiple real questions

Runs a small set of questions end-to-end in one pass so you can eyeball answers and sources together, instead of re-running §8 one question at a time. Replace the list with questions relevant to your own documents.

In [12]:
test_questions = [
    "What is this project about?",
    "What does Phase-1 of the RAG pipeline involve?",
    "What tools are used for reranking in Phase-2?",
    "What is RAG?"
]

for q in test_questions:
    answer, sources = answer_query(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"Sources: {[d.metadata.get('source') for d in sources]}")
    print("-" * 80)

Q: What is this project about?
A: The project involves building and evaluating machine‑learning, deep‑learning, and natural‑language‑processing models in Python for predictive‑analytics and classification tasks, and also creating a computer‑vision‑based face‑recognition system that uses OpenCV together with deep‑learning techniques【1】.
Sources: ['..\\data\\resume.pdf', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation']
--------------------------------------------------------------------------------
Q: What does Phase-1 of the RAG pipeline involve?
A: Phase‑1 of a Retrieval‑augmented Generation (RAG) pipeline is the **ingestion and indexing stage**. In this phase the system:

1. **Ingests source documents** such as PDFs, Markdown files, and web pages.  
2. **Chunks the text** into mana